# 🏆 Day 1 Gauntlet — Titanic Master Challenge

This is the **capstone challenge** for Pandas Day 1. It combines every topic from the day into a single end-to-end data analysis workflow on the real Titanic dataset (891 passengers, 12 columns).

## Topics exercised
| Task | Skills |
|------|--------|
| 1 — Basic Inspection | `df.shape`, `.columns`, `.dtypes` |
| 2 — Type Conversion | `.astype(bool)` |
| 3 — New Columns | Vectorized arithmetic, boolean comparison |
| 4 — Rename & Drop | `.rename()`, `.drop()` |
| 5 — Filtering | Boolean masking with `&` |
| 6 — Sorting | `.sort_values()`, `.head()` |
| 7 — Groupby Aggregation | `.groupby()`, `.mean()` |
| 8 — Survival Rate | Boolean mean as a percentage |

## The Dataset
The Titanic dataset is a classic for data analysis because it has:
- Mixed dtypes (numeric, categorical, text)
- Missing values (`Age`, `Cabin`, `Embarked`)
- A clear binary target (`Survived`)
- Interesting real-world patterns (class, gender, family effects on survival)


In [1]:
#Gauntlet #1 – Day 1 Master Challenge (Titanic Edition)

## Setup — Load and Begin

The dataset is loaded directly from GitHub via URL — `pd.read_csv()` accepts any URL, not just local paths.

### What each task builds on the previous
Tasks 3 and 4 **modify the DataFrame** (`df`). Tasks 5–8 use those modified columns.  
So the order of execution matters — run cells top to bottom.


### Task 1 — Basic Inspection
`.shape` → 891 rows, 12 columns  
`.columns.tolist()` → see all column names  
`.dtypes.head(5)` → check the first 5 dtypes: `PassengerId`, `Survived`, `Pclass` are `int64`; `Name`, `Sex` are `object`

### Task 2 — Convert 'Survived' to Boolean
`Survived` is stored as `0`/`1` integers. Converting to `bool` makes the semantics explicit:  
- `False` = did not survive  
- `True` = survived  
This also enables `.mean()` as a survival rate: `True=1, False=0` → mean = fraction who survived.

### Task 3 — Feature Engineering: Three New Columns
```python
FamilySize   = SibSp + Parch + 1       # count of all family members incl. self
IsAlone      = FamilySize == 1         # boolean: True if traveling alone
FarePerPerson = Fare / FamilySize      # normalize fare by family size
```
These derived features often have more predictive power than the raw columns.

### Task 4 — Rename and Drop
- Rename `Pclass` → `Class`, `SibSp` → `Siblings_Spouses` for clarity
- Drop `Ticket` (arbitrary string, no analytical value) and `Cabin` (>75% missing)

### Task 5 — Filtering: First Class Women
`(df['Class'] == 1) & (df['Sex'] == 'female')` — AND filter using modified column names from Task 4.

### Task 6 — Sorting: 5 Youngest Passengers
`.sort_values('Age')` sorts ascending by default → smallest age first.  
`.head(5)` takes the first 5 rows. Note: passengers with missing `Age` are moved to the end.

### Task 7 — GroupBy: Average Fare by Survival
`.groupby('Survived')['Fare'].mean()` splits the DataFrame into survivors and non-survivors, then computes the mean fare for each group.  
Result: survivors paid significantly higher fares on average — a classic class-survival correlation.

### Task 8 — Survival Rate Calculation
`first_class_women['Survived'].mean() * 100`  
Because `Survived` is a boolean column, `.mean()` = fraction of `True` values = survival rate.  
**~96.8%** of first-class women survived — one of the strongest signals in the dataset.


In [14]:
import pandas as pd
import numpy as np

# Load Titanic dataset
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# ============================================================================
# Task 1: Basic Inspection
# ============================================================================
print("=== Task 1: Basic Inspection ===")
print(f"Rows, Columns: {df.shape}")
print("Column names:", df.columns.tolist())
print("\nData types of first 5 columns:")
print(df.dtypes.head(5))

# ============================================================================
# Task 2: Convert 'Survived' to boolean
# ============================================================================
df['Survived'] = df['Survived'].astype(bool)
print("\n=== Task 2: Survived as bool ===")
print(df['Survived'].dtype)

# ============================================================================
# Task 3: Create New Columns
# ============================================================================
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = df['FamilySize'] == 1
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
print("\n=== Task 3: New Columns ===")
print(df[['SibSp', 'Parch', 'FamilySize', 'IsAlone', 'Fare', 'FarePerPerson']].head())

# ============================================================================
# Task 4: Rename & Drop
# ============================================================================
df = df.rename(columns={'Pclass': 'Class', 'SibSp': 'Siblings_Spouses'})
df = df.drop(['Ticket', 'Cabin'], axis=1)
print("\n=== Task 4: Renamed & Dropped ===")
print("Remaining columns:", df.columns.tolist())

# ============================================================================
# Task 5: Filtering - First Class Women
# ============================================================================
first_class_women = df[(df['Class'] == 1) & (df['Sex'] == 'female')]
print("\n=== Task 5: First Class Women ===")
print(f"Number of first class women: {len(first_class_women)}")

# ============================================================================
# Task 6: Sorting - 5 Youngest Passengers
# ============================================================================
youngest = df.sort_values('Age').head(5)
print("\n=== Task 6: 5 Youngest Passengers ===")
print(youngest[['Name', 'Age']])

# ============================================================================
# Task 7: Average Fare by Survival
# ============================================================================
avg_fare = df.groupby('Survived')['Fare'].mean()
print("\n=== Task 7: Average Fare by Survival ===")
print(f"Non-Survivors: ${avg_fare[False]:.2f}")
print(f"Survivors:     ${avg_fare[True]:.2f}")

# ============================================================================
# Task 8 (Bonus): Survival Rate of First Class Women
# ============================================================================
survival_rate = first_class_women['Survived'].mean() * 100
print("\n=== Task 8 (Bonus): First Class Women Survival Rate ===")
print(f"{survival_rate:.1f}%")

=== Task 1: Basic Inspection ===
Rows, Columns: (891, 12)
Column names: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Data types of first 5 columns:
PassengerId     int64
Survived        int64
Pclass          int64
Name           object
Sex            object
dtype: object

=== Task 2: Survived as bool ===
bool

=== Task 3: New Columns ===
   SibSp  Parch  FamilySize  IsAlone     Fare  FarePerPerson
0      1      0           2    False   7.2500        3.62500
1      1      0           2    False  71.2833       35.64165
2      0      0           1     True   7.9250        7.92500
3      1      0           2    False  53.1000       26.55000
4      0      0           1     True   8.0500        8.05000

=== Task 4: Renamed & Dropped ===
Remaining columns: ['PassengerId', 'Survived', 'Class', 'Name', 'Sex', 'Age', 'Siblings_Spouses', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'FarePerPerson']

=== Task 5: Firs